# prep.tableS2

In [35]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import taxoniq
plt.rcParams['svg.fonttype'] = 'none'
from yaml import load, Loader
from daforfer import DaforferDB
conf = load(open("conf.yaml"), Loader)
db = DaforferDB(conf['database'])
si = DaforferDB(conf['si'])
si.toc()

┌─────────┬─────────────────────────────────────┐
│  name   │             description             │
│ varchar │               varchar               │
├─────────┼─────────────────────────────────────┤
│ TableS1 │ Table S1: Library sites and context │
└─────────┴─────────────────────────────────────┘

In [ ]:
bacteria_hits = db.conn.sql('SELECT * FROM D_bacteriaHits').df()
metadata = db.conn.sql('SELECT * FROM D_sites').df()
bacteria_hits = pd.merge(metadata, bacteria_hits, on='library', how='left').dropna(subset='taxid')
bacteria_hits['taxid'] = bacteria_hits['taxid'].astype(int)
bacteria_hits

,site,library,habitat,n_extracts,host_taxon,taxid,scientific_name,is_pab,pab_type
0,C1,PV534,Crop,3,Diplotaxis erucoides,59620,uncultured Clostridium sp.,False,
1,C1,PV535,Crop,17,Brassica oleracea,1563157,Pseudomonas endophytica,True,pab_unknown
2,C1,PV535,Crop,17,Brassica oleracea,1270,Micrococcus luteus,True,pab_unknown
3,C1,PV535,Crop,17,Brassica oleracea,59620,uncultured Clostridium sp.,False,
4,C1,PV538,Crop,8,Brassica oleracea,82995,Serratia grimesii,False,
...,...,...,...,...,...,...,...,...,...
1512,Z2,PV529,Crop,1,Picris echioides,289370,Pseudomonas argentinensis,True,pab_unknown
1513,Z2,PV529,Crop,1,Picris echioides,1736265,Methylobacterium sp. Leaf125,True,pab_unknown
1514,Z2,PV529,Crop,1,Picris echioides,91844,Candidatus Portiera aleyrodidarum,False,
1515,Z2,PV529,Crop,1,Picris echioides,1192054,Paracoccus sp. 228,False,


## Host range 


First, we will compute the host range, using the `host_taxon`, `taxid` and `scientific_name` as keys, and then counting by `scientific_name` and `taxid`. This should tell us how many different hosts can each entry find.  

In [37]:
bacteria_host_range = bacteria_hits.query('is_pab == True').value_counts(
    ['host_taxon', 'scientific_name', 'taxid']
    ).reset_index().value_counts(
        ['scientific_name', 'taxid']
    ).reset_index().rename(columns={'count': 'host_range'})
assert(len(bacteria_host_range) == 127) # INTRODUCING THIS KIND OF CHECKS CAN HELP US CHECKING OUR OPERATIONS
bacteria_host_range

,scientific_name,taxid,host_range
0,Frigoribacterium sp. Leaf164,1736282,22
1,Pseudomonas lutea,243924,16
2,Rhodococcoides fascians,1828,15
3,Methylobacterium sp. Leaf125,1736265,14
4,Duffyella gerundensis,1619313,14
...,...,...,...
122,Erwinia mallotivora,69222,1
123,Enterobacter kobei,208224,1
124,Duganella sp. CF458,1884368,1
125,Duganella sp. CF402,1855289,1


## Hits per habitat

Now, we will count the number of hits per habitat. To do so, we simply do a value counts of the hits sharing `taxid,scientific_name,habitat`, and then we pivot the dataset to get it columns.

In [38]:
bacteria_pab_perhabitat = bacteria_hits.query('is_pab == True').value_counts(["taxid", "scientific_name", "habitat"]).reset_index() 
bacteria_pab_perhabitat = bacteria_pab_perhabitat.pivot(index=['scientific_name', 'taxid'], columns='habitat', values='count').fillna(0).reset_index()
assert(len(bacteria_pab_perhabitat) == 127)
bacteria_pab_perhabitat


habitat,scientific_name,taxid,Crop,Edge,Oak,Wasteland
0,Achromobacter xylosoxidans,85698,1.0,1.0,0.0,0.0
1,Acidovorax sp. Leaf160,1736280,0.0,0.0,0.0,1.0
2,Acidovorax sp. Leaf78,1736237,0.0,0.0,0.0,1.0
3,Acinetobacter baumannii,470,1.0,0.0,0.0,0.0
4,Acinetobacter pittii,48296,2.0,1.0,0.0,0.0
...,...,...,...,...,...,...
122,Stutzerimonas stutzeri,316,0.0,1.0,0.0,1.0
123,Variovorax paradoxus,34073,0.0,0.0,1.0,1.0
124,Xanthomonas campestris,339,0.0,4.0,0.0,4.0
125,Xaviernesmea rhizosphaerae,1672749,1.0,0.0,0.0,0.0


## Host-range by habitat

A slightly more complex query consists on getting the host-ranges per habitat. To do so, we need to first deduplicate the bacteria-hit dataset by `scientific_name,taxid,host_taxon,habitat`, and then repeat. 

In [39]:
bacteria_pab_hrperhabitat = bacteria_hits.query('is_pab == True').drop_duplicates(
    ["taxid", "scientific_name", "host_taxon", "habitat"]
).value_counts(
    ["taxid", "scientific_name", "habitat"]
).reset_index() 
bacteria_pab_hrperhabitat = bacteria_pab_hrperhabitat.pivot(index=['scientific_name', 'taxid', ], columns='habitat', values='count').fillna(0).reset_index()
assert(len(bacteria_pab_hrperhabitat) == 127)
bacteria_pab_hrperhabitat


habitat,scientific_name,taxid,Crop,Edge,Oak,Wasteland
0,Achromobacter xylosoxidans,85698,1.0,1.0,0.0,0.0
1,Acidovorax sp. Leaf160,1736280,0.0,0.0,0.0,1.0
2,Acidovorax sp. Leaf78,1736237,0.0,0.0,0.0,1.0
3,Acinetobacter baumannii,470,1.0,0.0,0.0,0.0
4,Acinetobacter pittii,48296,2.0,1.0,0.0,0.0
...,...,...,...,...,...,...
122,Stutzerimonas stutzeri,316,0.0,1.0,0.0,1.0
123,Variovorax paradoxus,34073,0.0,0.0,1.0,1.0
124,Xanthomonas campestris,339,0.0,3.0,0.0,4.0
125,Xaviernesmea rhizosphaerae,1672749,1.0,0.0,0.0,0.0


## Site-ranges

Now, we will compute site-ranges: the number of different sites that are reached by each bacteria.

In [40]:
bacteria_pab_sr = bacteria_hits.query('is_pab == True').drop_duplicates(
    ["taxid", "scientific_name", "site"]
).value_counts(
    ["taxid", "scientific_name"]
).reset_index().rename(columns={'count': 'site_range'})
# bacteria_pab_sr = bacteria_pab_sr.pivot(index=['scientific_name', 'taxid', ], columns='site', values='count').fillna(0).reset_index()
assert(len(bacteria_pab_sr) == 127)
bacteria_pab_sr


,taxid,scientific_name,site_range
0,243924,Pseudomonas lutea,13
1,1736282,Frigoribacterium sp. Leaf164,10
2,1828,Rhodococcoides fascians,9
3,1736265,Methylobacterium sp. Leaf125,9
4,1619313,Duffyella gerundensis,7
...,...,...,...
122,1144319,Herbaspirillum sp. CF444,1
123,1144310,Rhizobium sp. CF080,1
124,1028989,Pseudomonas sp. StFLB209,1
125,582680,Microbacterium azadirachtae,1


## Habitat-range

Finally, we compute the number of habitats

In [41]:
bacteria_pab_habitatr = bacteria_hits.query('is_pab == True').drop_duplicates(
    ["taxid", "scientific_name", "habitat"]
).value_counts(
    ["taxid", "scientific_name"]
).reset_index().rename(columns={'count': 'habitat_range'})
# bacteria_pab_sr = bacteria_pab_sr.pivot(index=['scientific_name', 'taxid', ], columns='site', values='count').fillna(0).reset_index()
assert(len(bacteria_pab_habitatr) == 127)
bacteria_pab_habitatr


,taxid,scientific_name,habitat_range
0,1736265,Methylobacterium sp. Leaf125,4
1,243924,Pseudomonas lutea,4
2,1619313,Duffyella gerundensis,4
3,1736282,Frigoribacterium sp. Leaf164,4
4,2035,Curtobacterium flaccumfaciens,4
...,...,...,...
122,1230476,Bradyrhizobium sp. DFCI-1,1
123,1144319,Herbaspirillum sp. CF444,1
124,1144310,Rhizobium sp. CF080,1
125,1028989,Pseudomonas sp. StFLB209,1


## Merge

Now, we can try to merge it all.

In [42]:
pab_distribution = pd.merge(
    bacteria_hits.value_counts(['scientific_name', 'taxid', 'pab_type']).reset_index().rename(columns={'count': 'hits'}),
    bacteria_host_range,
    on=['taxid', 'scientific_name']
)
pab_distribution = pd.merge(
    pab_distribution, bacteria_pab_habitatr, on=['taxid', 'scientific_name']
)

pab_distribution = pd.merge(
    pab_distribution, bacteria_pab_sr, on=['taxid', 'scientific_name']
)

pab_distribution = pd.merge(
    pab_distribution, bacteria_pab_hrperhabitat.rename(columns={'Crop': 'Crop_HR', 'Edge': 'Edge_HR', 'Oak': 'Oak_HR', 'Wasteland': 'Wasteland_HR'}), 
    on=['taxid', 'scientific_name']
)

pab_distribution = pd.merge(
    pab_distribution, bacteria_pab_perhabitat.rename(columns={'Crop': 'Crop_hits', 'Edge': 'Edge_hits', 'Oak': 'Oak_hits', 'Wasteland': 'Wasteland_hits'}), 
    on=['taxid', 'scientific_name']
)

assert(len(pab_distribution)==127)
pab_distribution['pab_type'] = pab_distribution['pab_type'].apply(
    lambda x: {'pab_unknown': 'U', 'pab_symbiont': 'M', 'pab_pathogen': 'P'}[x]
)
pab_distribution

,scientific_name,taxid,pab_type,hits,host_range,habitat_range,site_range,Crop_HR,Edge_HR,Oak_HR,Wasteland_HR,Crop_hits,Edge_hits,Oak_hits,Wasteland_hits
0,Frigoribacterium sp. Leaf164,1736282,U,26,22,4,10,2.0,11.0,1.0,10.0,2.0,13.0,1.0,10.0
1,Pseudomonas lutea,243924,U,21,16,4,13,1.0,5.0,8.0,6.0,1.0,5.0,9.0,6.0
2,Rhodococcoides fascians,1828,U,21,15,3,9,3.0,11.0,0.0,3.0,3.0,15.0,0.0,3.0
3,Methylobacterium sp. Leaf125,1736265,U,17,14,4,9,1.0,4.0,3.0,8.0,1.0,4.0,4.0,8.0
4,Chryseobacterium sp. Leaf201,1735672,U,15,13,2,6,0.0,4.0,0.0,10.0,0.0,4.0,0.0,11.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,Azospirillum lipoferum,193,U,1,1,1,1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
123,Arthrobacter sp. OY3WO11,1835723,U,1,1,1,1,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
124,Arthrobacter sp. SPG23,1610703,U,1,1,1,1,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
125,Microbacterium ginsengisoli,400772,U,1,1,1,1,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


#### Testing

We have performed many operations at once. It is therefore convenient to check that the values that we are observing add up. Below, we will run a few tests.

In [43]:
# TEST 1: No organism should have a host range larger than the host range that we had measured 
# at one of the habitats. 
for i in range(len(pab_distribution)):
    example = pab_distribution.loc[i]
    try:
        assert(max([example.Crop_HR,example.Edge_HR, example.Oak_HR,example.Wasteland_HR]) <= example.host_range)
    except AssertionError: 
        print("{0} did not pass the test 1".format(example['scientific_name']))

In [44]:
for i in range(len(pab_distribution)):
    example = pab_distribution.loc[i]
    example_subset = bacteria_hits.query("taxid == {0}".format(example.taxid))
    assert(len(example_subset) == int(example.Crop_hits + example.Edge_hits + example.Wasteland_hits + example.Oak_hits))
    assert(len(example_subset) == int(example.hits))
    assert(len(example_subset.drop_duplicates(['host_taxon'])) == example.host_range)
print("All tests passed! :-D")

All tests passed! :-D


## Save

In [45]:
db.save_dataframe(
    pab_distribution, table_name="D_PABOTUs",
    description="This table summarizes most of the information of our detected OTUs, including host_range, site_range, habitat_range, etc."
)

Saved D_PABOTUs to db.2025-10-27


In [46]:
si.save_dataframe(
    pab_distribution, table_name="TableS2",
    description="This table summarizes most of the information of our detected OTUs, including host_range, site_range, habitat_range, etc."
)

Saved TableS2 to si.2025-10-27


In [47]:
db.conn.close()
si.conn.close()